In [ ]:
import requests, json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import Counter

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

API = "http://localhost:8080"

# Load all cache panels once
cache = requests.get(API + "/api/cache/all").json()
overview = cache["overview"]
timeline = cache["timeline"]
cities = cache["cities"]
keywords = cache["keywords"]
official = cache["official"]
youtube = cache["youtube"]

print("API connected. Panels loaded:", list(cache.keys()))
print("Total housing_posts:", overview["total_docs"])


# Housing Insecurity in Australian Online Communities
## COMP90024 Assignment 2 — Team 12

This notebook is the frontend demonstration for our multi-platform housing insecurity analytics system. 
It connects live to the deployed Flask REST API and presents cross-platform discussion volume, 
sentiment patterns, keyword analysis, timeline trends, YouTube analytics, and official data comparison.

**Data sources**: BlueSky · Mastodon · GDELT · YouTube + Official housing data


# 1. System Overview

The frontend is the presentation layer of a cloud-based data pipeline deployed on MRC/NeCTAR Kubernetes.

```text
BlueSky / Mastodon / GDELT / YouTube
    -> Harvesting (Fission Timers + K8s Deployment + CronJob)
    -> Redis MQ (BlueSky/Mastodon) + Direct ES (GDELT/YouTube)
    -> Normalisation + Sentiment (244-entry domain lexicon + TextBlob)
    -> Elasticsearch Storage
    -> Flask REST API (28 endpoints, CORS-enabled)
    -> Jupyter Notebook Frontend
```

| Stage | Technology |
|---|---|
| Collection | Fission Timers (@every 30m), K8s Deployment (GDELT @15m), CronJob (YouTube daily) |
| Message Queue | Redis + KEDA event-driven autoscaling (BlueSky/Mastodon) |
| Processing | Domain lexicon sentiment (244 entries, 35% TextBlob / 65% domain) |
| Storage | Elasticsearch 9.2.4, 2-node cluster, 5 indices |
| API | Flask, 4 blueprints, 28 endpoints, CORS |
| Frontend | Jupyter Notebook (this dashboard) |


# 2. Cross-platform Discussion Volume

Compares housing-related documents across all four public discussion platforms.
YouTube count includes both videos and associated comments as discussion units.


In [ ]:
vol = requests.get(API + "/api/housing/volume-by-platform").json()
platforms = [r["platform"] for r in vol["results"]]
counts = [r["post_count"] for r in vol["results"]]

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
plt.figure(figsize=(10, 5))
bars = plt.bar(platforms, counts, color=colors)
plt.title("Cross-platform Discussion Volume", fontsize=14, fontweight="bold")
plt.ylabel("Documents / Discussion Units")

for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f"{count:,}", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/platform_volume.png", dpi=200, bbox_inches="tight")
plt.show()


### Interpretation

BlueSky and GDELT contribute the largest share of indexed records. YouTube shows 335,533 discussion units 
(6,039 videos + 329,494 comments). Mastodon provides federated social discussion primarily from 
Australian instances (aus.social, mastodon.au).


# 3. Platform Sentiment Comparison

Compares sentiment patterns across platforms. Sentiment is computed via a unified 244-entry 
domain housing lexicon with TextBlob blending (35% TextBlob / 65% domain).


In [ ]:
sent = requests.get(API + "/api/analysis/housing/sentiment-by-platform").json()
platforms_s, neg_vals, neu_vals, pos_vals = [], [], [], []

for b in sent.get("buckets", []):
    platforms_s.append(b["platform"])
    labels = {lb["sentiment_label"]: lb["count"] for lb in b["sentiment_labels"]}
    total = sum(labels.values())
    neg_vals.append(labels.get("negative", 0) / total * 100)
    neu_vals.append(labels.get("neutral", 0) / total * 100)
    pos_vals.append(labels.get("positive", 0) / total * 100)

x = np.arange(len(platforms_s))
w = 0.22
colors_s = ["#C44E52", "#8C8C8C", "#55A868"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, neg_vals, w, label="Negative", color=colors_s[0])
ax.bar(x, neu_vals, w, label="Neutral", color=colors_s[1])
ax.bar(x + w, pos_vals, w, label="Positive", color=colors_s[2])
ax.set_xticks(x)
ax.set_xticklabels(platforms_s)
ax.set_title("Platform Sentiment Comparison", fontsize=14, fontweight="bold")
ax.set_ylabel("Percentage (%)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/platform_sentiment.png", dpi=200, bbox_inches="tight")
plt.show()


### Interpretation

YouTube shows the strongest negative share in the sample, which is plausible given that 
comments often express frustration or lived experience. BlueSky and Mastodon are dominated
by neutral short-form posts. GDELT news articles span a wider sentiment range.
Sentiment is an approximate signal of tone, not a direct measurement of public experience.


In [ ]:
# Word cloud
# First try live generation, fall back to pre-saved local image

try:
    from wordcloud import WordCloud
    wc = requests.get(API + "/api/analysis/housing/word-cloud", params={"word_size": 100}).json()
    wc_data = {w["text"]: w["value"] for w in wc.get("words", []) if w["text"] and len(w["text"]) > 2}

    wordcloud = WordCloud(width=900, height=400, background_color="white",
                          colormap="viridis", max_words=100, collocations=False)
    wordcloud.generate_from_frequencies(wc_data)
    plt.figure(figsize=(12, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Housing Discussion Word Cloud", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print("Word cloud generated live from API")
except ImportError:
    from IPython.display import Image
    display(Image("figures/wordcloud.png"))
    print("Word cloud loaded from figures/wordcloud.png")
except Exception as e:
    from IPython.display import Image
    display(Image("figures/wordcloud.png"))
    print("Word cloud loaded from local file (API unavailable)")


In [ ]:
kw = keywords.get("top_keywords", [])[:15]
kws = [k["keyword"] for k in kw]
kcounts = [k["count"] for k in kw]

plt.figure(figsize=(10, 5))
plt.barh(range(len(kws)), kcounts, color="#4C72B0")
plt.yticks(range(len(kws)), kws)
plt.title("Top 15 Keywords Across All Platforms", fontsize=14, fontweight="bold")
plt.xlabel("Document Count")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/keywords.png", dpi=200, bbox_inches="tight")
plt.show()


Keywords reflect GDELT theme codes (econ_housing_prices, crisislex_c05_need_of_shelters)
dominating the dataset, followed by YouTube and social media query keywords.


# 5. Word Cloud

Visualises the most prominent terms extracted from recent housing-related discussion text.


In [ ]:
try:
    from wordcloud import WordCloud
    wc = requests.get(API + "/api/analysis/housing/word-cloud", params={"word_size": 100}).json()
    wc_data = {w["text"]: w["value"] for w in wc.get("words", []) if w["text"] and len(w["text"]) > 2}

    wordcloud = WordCloud(width=900, height=400, background_color="white",
                          colormap="viridis", max_words=100, collocations=False)
    wordcloud.generate_from_frequencies(wc_data)
    plt.figure(figsize=(12, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Housing Discussion Word Cloud", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("figures/wordcloud.png", dpi=200, bbox_inches="tight")
    plt.show()
except ImportError:
    print("pip install wordcloud")
    wc = requests.get(API + "/api/analysis/housing/word-cloud", params={"word_size": 30}).json()
    for w in wc.get("words", [])[:20]:
        if w.get("text"):
            print(f"  {w['text']:30s} {w['value']}")


# 6. Discussion Timeline

Shows daily discussion volume over the past 30 days with sentiment breakdown.


In [ ]:
tl = timeline.get("timeline", [])
dates = [d["date"] for d in tl]
totals = [d["total"] for d in tl]

plt.figure(figsize=(12, 5))
plt.plot(dates, totals, marker="o", linewidth=2, color="#4C72B0")
plt.fill_between(range(len(dates)), totals, alpha=0.1, color="#4C72B0")
plt.title("Daily Discussion Volume — 30 Day Trend", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Documents")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/timeline.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Date range: {dates[0]} to {dates[-1]}")
print(f"Average daily volume: {sum(totals)//len(totals):,} docs")
print(f"Latest day: {totals[-1]:,} docs")


# 7. YouTube Analytics

Pre-computed YouTube channel distribution, yearly trends, and engagement statistics.


In [ ]:
print("=" * 50)
print("YOUTUBE SUMMARY")
print("=" * 50)
print(f"Total videos:      {youtube.get('total_videos', 0):>10,}")
print(f"Total comments:    {youtube.get('total_comments', 0):>10,}")
print(f"Discussion units:  {youtube.get('total_discussion_units', 0):>10,}")
print(f"Total likes:       {youtube.get('like_stats', {}).get('total', 0):>10,.0f}")
print()

# Top channels
channels = youtube.get("top_channels", [])[:12]
chnames = [c["channel"] for c in channels]
chcounts = [c["videos"] for c in channels]

plt.figure(figsize=(10, 5))
plt.barh(range(len(chnames)), chcounts, color="#55A868")
plt.yticks(range(len(chnames)), chnames)
plt.title("Top YouTube Channels Discussing Housing", fontsize=14, fontweight="bold")
plt.xlabel("Number of Videos")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/youtube_channels.png", dpi=200, bbox_inches="tight")
plt.show()

# Yearly trend
yt_trend = youtube.get("yearly_trend", [])
years_yt = [t["year"] for t in yt_trend if t["year"]]
vids_yt = [t["videos"] for t in yt_trend if t["year"]]

plt.figure(figsize=(10, 5))
plt.plot(years_yt, vids_yt, marker="o", linewidth=2, color="#C44E52")
plt.title("YouTube Housing-related Videos Over Time", fontsize=14, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Number of Videos")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/youtube_trend.png", dpi=200, bbox_inches="tight")
plt.show()


# 8. Official Housing Data Comparison

Official data provides real-world context. This section compares online discussion volume 
with official housing indicators to show how public attention aligns with structured data.


In [ ]:
print("=" * 50)
print("OFFICIAL DATA SUMMARY")
print("=" * 50)
print(f"Total records: {official.get('total_docs', 0):,}")
print()

# By state
by_state = official.get("by_state", {})
states = sorted(by_state.items(), key=lambda x: x[1], reverse=True)[:10]
st_names = [s[0] for s in states]
st_counts = [s[1] for s in states]

plt.figure(figsize=(10, 5))
plt.bar(st_names, st_counts, color="#8172B2")
plt.title("Official Housing Data by State", fontsize=14, fontweight="bold")
plt.ylabel("Number of Records")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/official_state.png", dpi=200, bbox_inches="tight")
plt.show()

# By source
by_source = official.get("by_source", {})
sources = sorted(by_source.items(), key=lambda x: x[1], reverse=True)[:8]
src_names = [s[0][:30] for s in sources]
src_counts = [s[1] for s in sources]

plt.figure(figsize=(10, 5))
plt.barh(range(len(src_names)), src_counts, color="#4C72B0")
plt.yticks(range(len(src_names)), src_names)
plt.title("Official Data Sources", fontsize=14, fontweight="bold")
plt.xlabel("Number of Records")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/official_sources.png", dpi=200, bbox_inches="tight")
plt.show()


### Interpretation

Official data is used as comparative context, not merged with social discussion records. 
NSW Rental Bonds dominate the dataset (5.96M records), followed by ABS Census and WA/SA/TAS data. 
Online discussion and official indicators are produced through different processes — 
the comparison is at the aggregate level only and does not imply causation.


# 9. Sentiment Distribution

Overall sentiment across all 1.7M+ housing-related documents.


In [ ]:
sent_dist = overview.get("sentiment_distribution", {})
labels = list(sent_dist.keys())
values = list(sent_dist.values())
colors_pie = ["#C44E52", "#8C8C8C", "#55A868"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.pie(values, labels=labels, autopct="%1.1f%%", colors=colors_pie, 
        explode=(0.02, 0, 0.02), startangle=90)
ax1.set_title("Overall Sentiment Distribution", fontsize=13, fontweight="bold")

ax2.bar(labels, values, color=colors_pie)
ax2.set_title("Sentiment Count by Category", fontsize=13, fontweight="bold")
ax2.set_ylabel("Document Count")
for i, v in enumerate(values):
    ax2.text(i, v + 5000, f"{v:,}", ha="center", fontsize=11, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("figures/sentiment_overall.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Negative: {values[0]:,} ({values[0]/sum(values)*100:.1f}%)")
print(f"Neutral:  {values[1]:,} ({values[1]/sum(values)*100:.1f}%)")
print(f"Positive: {values[2]:,} ({values[2]/sum(values)*100:.1f}%)")


# 10. Conclusion

## Key Findings

- **Rental stress dominates**: Rent, housing crisis, and affordability are the most visible themes across all platforms.
- **Platform differences are clear**: YouTube shows stronger negative sentiment (29.0%), while BlueSky and Mastodon are predominantly neutral short-form posts. GDELT provides balanced news coverage.
- **Scale is meaningful**: 1.7M+ social/news documents across 4 platforms, plus 7.3M official records, with live data collection and a working cloud pipeline.
- **Official data provides context**: NSW rental bond data (5.96M records) shows the real-world scale of rental activity against which online discussion can be compared.

## Limitations

- Online platforms are not representative of all Australians.
- Sentiment is an approximate signal, not ground truth.
- Location inference from text is incomplete.
- Official data and online discussion are different types of evidence and should not be merged uncritically.

## System Architecture

The full system includes: MRC/NeCTAR cloud deployment, Kubernetes orchestration, Fission serverless functions, 
Redis + KEDA event-driven message queue, Elasticsearch storage, Flask REST API, and this Jupyter Notebook frontend.
All components are verified with runtime evidence.
